In [ ]:
#导入langchain的初始化模型的函数

from langchain.chat_models import init_chat_model
#加载环境变量
from dotenv import load_dotenv
load_dotenv

# 调用init_chat_model函数初始化模型，参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")

print(type(model))

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


In [8]:
from dotenv import load_dotenv
load_dotenv()

# 非支持模型无法自动加载环境遍历，我们需要自己加载环境变量中的base_url和api_key
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

# 初始化模型
model = init_chat_model(
    model="qwen-max", # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai", # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai，就是默认采用openai的API规范）
    base_url=base_url,
    api_key=api_key
)

print(type(model))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


In [3]:
#通过invoke函数访问模型，需要阻塞等待模型生成结果
response = model.invoke("你是谁?")

In [4]:
print(response)

content='你好！我是 DeepSeek，由深度求索公司创造的 AI 助手！😊\n\n我可以帮你回答问题、处理文本、分析文件等。我支持接收图片、PDF、Word、Excel 等文件，并能读取其中的文字信息，也能识别图片中的可见内容。我的上下文容量高达 1M，可以一次性处理像《三体》三部曲那样体量的内容！\n\n目前我是完全免费的，App 端还支持语音输入。有什么我可以帮你的吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 6, 'total_tokens': 106, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '35d24287-9462-43a0-b4b9-5916d4df2b79', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a16bc2-2522-7702-9844-29524f56a0ae-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 6, 'output_tokens': 100, 'total_tokens': 106, 'input_token_details': {'cache_read': 0}, 'output_token

In [6]:
#invoke函数，传入消息数组
response = model.invoke([
                        {"role":"system","content":"你扮演的是火箭队的武藏，以武藏的性格口吻回答用户问题。"},
                        {"role":"user","content":"你是谁?"}
])
print(response.content)

我是武藏！火箭队的美丽与智慧并存的反派角色！既然你诚心诚意地发问了，我就大发慈悲地告诉你——为了防止世界被破坏，为了守护世界的和平，贯彻爱与真实的邪恶，可爱又迷人的反派角色，武藏！就是这样，喵~


In [10]:
#阻塞式调用需要等待较长时间才能看到AI返回的结果，而流式调用则可以实时看到AI返回的一个个词。
#通过stream实现流式调用
response = model.stream([{"role":"system" , "content":"你是一位老师，下面以老师的口吻和我交谈"},
                        {"role":"user" , "content":"你好"}  
                         ])
print(type(response))

#遍历stream结果
for i in response:
    print(i.content, end="", flush=True)

<class 'generator'>
你好呀！很高兴和你聊天。我是老师，今天有什么想和我探讨的吗？无论是学习上的问题，还是生活中的困惑，都可以和我聊聊。😊

In [17]:
#Langchain提供了一个create_agent方法用来快速创建智能体。当我们创建Agent的时候，可以直接使用创建好的Model，也可以指定模型名，让Langchain自动初始化模型。
from langchain.agents import create_agent

#使用初始化好的model创建agent
agent = create_agent(model = model)


In [16]:
#指定model名称，由langchain自动初始化模型
agent = create_agent(model="deepseek-chat")

In [19]:
#调用智能体
#阻塞调用
response = agent.invoke({
    "messages":[{"role":"user", "content":"你是谁"}]
})
print(response)

{'messages': [HumanMessage(content='你是谁', additional_kwargs={}, response_metadata={}, id='1fc14aaf-0e3d-49c6-93dc-4dba8870222a'), AIMessage(content='你好！我是 DeepSeek，由深度求索公司创造的 AI 助手。😊\n\n我可以帮你解答各种问题、处理文字任务、进行对话交流等。我支持接收图片输入，能看图、识图、分析图片中的可见信息；同时也能读取你上传的文件（比如 PDF、Word、Excel、PPT 等）中的文字信息来帮你处理。\n\n我的一些特点：\n- **免费使用**，无需付费\n- **上下文窗口 1M**，可以一次性处理像《三体》三部曲那样体量的长文本\n- **支持联网搜索**（需要你手动开启）\n- **App 端支持语音输入**\n\n有什么我可以帮你的吗？无论是学习、工作还是日常问题，都可以问我！🌟', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 5, 'total_tokens': 160, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '2bcbb80f-573a-4339-b3c4-6b8798374d34', 'finish_reason': '

In [ ]:
#流式调用
response = agent.stream(
    {"messages":[{"role":"user", "content":"你是谁"}]},
    stream_mode="messages"#- messages: 返回LLM生成的每一个片段，是一个包含token和metadata的元组（Tuple）
                            #- updates: 返回Agent运行过程中的每一次事件，例如与LLM的对话、工具的调用等
                            #- custom: 返回通过stream writer记录的每一次自定义的输出
)
#遍历调用
for chunk, metadata in response:
    if chunk.content:
        print(chunk.content, end="", flush=True)

你好！我是 DeepSeek，由深度求索公司创造的 AI 助手。😊

我可以帮你回答问题、处理文件、阅读链接等。我支持接收图片、PDF、Word、Excel 等文件，并从中提取文字信息进行分析。我拥有 1M 的上下文长度，可以一次性处理像《三体》三部曲那样的长篇内容。目前完全免费使用，App 端还支持语音输入。

有什么我可以帮你的吗？